# Bangladesh baseline on Kaggle
Attach the reviewed prepared dataset as a private input. Add the read-only GitHub token as a Kaggle Secret named `RDD_GITHUB_TOKEN`. Enable internet and a GPU accelerator. See `docs/c2_baseline.md` for the data gate and the held-out test protocol.

In [ ]:
# Set these once for the attached private dataset and the commit you want to run.
INPUT_DIR = "/kaggle/input/your-bangladesh-dataset"
COMMIT = ""  # Exact 40-character Git commit containing the C2 code
REPO_URL = "https://github.com/MeazbahMehedi/road-damage-dg.git"
FOLD = 0
SEED = 1
EVAL_SPLIT = "val"
RESUME = False


In [ ]:
from pathlib import Path
import base64
import json
import os
import re
import shutil
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

work = Path("/kaggle/working")
repo = work / "road-damage-dg"
inputs = Path(INPUT_DIR)
root = work / "bangladesh"
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError("Set COMMIT to a full Git commit hash.")
frozen = json.loads((inputs / "Bangladesh/folds/plan.json").read_text())
review = json.loads((inputs / "Bangladesh/readiness.json").read_text())
if review.get("privacy_reviewed") is not True:
    raise ValueError("Use a shareable dataset with faces and plates reviewed before upload.")

def run(*args, **options):
    subprocess.run(args, check=True, **options)

if not repo.exists():
    token = UserSecretsClient().get_secret("RDD_GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}", "GIT_TERMINAL_PROMPT": "0"}
    run("git", "clone", REPO_URL, str(repo), env=env)
    del token, auth, env
run("git", "-C", str(repo), "checkout", "--detach", COMMIT)
run(sys.executable, "-m", "pip", "install", "-r", "requirements-c2.txt", cwd=repo)
run(sys.executable, "-m", "pip", "show", "torch", "ultralytics", cwd=repo)


In [ ]:
if not root.exists():
    shutil.copytree(inputs, root, ignore=shutil.ignore_patterns("folds", "*.cache"))
    holdout = root / "holdout_routes.txt"
    holdout.write_text("\n".join(frozen["holdout"]) + "\n", encoding="utf-8")
    extra = ["--leave-one-route-out"] if frozen["n_folds"] == frozen["n_routes"] - len(frozen["holdout"]) else []
    run(sys.executable, "-m", "rdd.data.make_folds",
        "--data-root", str(root), "--manifest", str(root / frozen["manifest"]),
        "--holdout-routes", str(holdout), "--n-folds", str(frozen["n_folds"]),
        "--seed", str(frozen["seed"]), *extra, cwd=repo)
rebuilt = json.loads((root / "Bangladesh/folds/plan.json").read_text())
if rebuilt != frozen:
    raise ValueError("Rebuilt folds differ from the reviewed plan.")


In [ ]:
import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle for the baseline.")
cfg = yaml.safe_load((repo / "configs/train/baseline_bd.yaml").read_text())
run_dir = work / "runs"
cfg.update(data_root=str(root), project_dir=str(run_dir), n_folds=frozen["n_folds"],
           fold=FOLD, seed=SEED, device="0")
config = work / "config.yaml"
config.write_text(yaml.safe_dump(cfg), encoding="utf-8")
resume_args = ["--resume"] if RESUME else []
run(sys.executable, "-m", "rdd.training.train", "--config", str(config),
    *resume_args, cwd=repo)
fold_id = "final" if FOLD == "final" else f"f{FOLD}"
exp_id = f"{Path(cfg['model']).stem}_Bangladesh_{fold_id}_s{SEED}"
results = work / "experiments/results/baseline.csv"
run(sys.executable, "-m", "rdd.evaluation.evaluate",
    "--weights", str(run_dir / exp_id / "weights/best.pt"), "--domain", "Bangladesh",
    "--split", EVAL_SPLIT, "--fold", str(FOLD), "--output", str(results), cwd=repo)
print(f"Kaggle outputs: {run_dir} and {results}")
